# 1. Creacion estructura medallion: Gold y Silver

In [0]:
-- Ejercicio 1.1
-- verificacion de catalogo
SHOW SCHEMAS IN bootcamp_de_valentin;

SHOW TABLES IN bootcamp_de_valentin.bronze;

SELECT COUNT(*) FROM bootcamp_de_valentin.bronze.propiedades_bronze;

In [0]:
-- Ejercicio 1.2
-- Crear schemas Silver y Gold
CREATE SCHEMA IF NOT EXISTS bootcamp_de_valentin.silver
COMMENT 'Schema silver para almacenar datos proceados, transformados y limpios. Bootcamp DE - Luciano Argolo';

CREATE SCHEMA IF NOT EXISTS bootcamp_de_valentin.gold
COMMENT 'Schema gold para almacenar datos modelados para el analisis de negocio con metricas y agregaciones necesarias. Bootcamp DE - Luciano Argolo';

In [0]:
SHOW SCHEMAS IN bootcamp_de_valentin

# 2. Crear Tabla Silver

**Ejercicio 2.1: Diseñar Tabla Silver**

Columnas necesarias basadas en el EDA de Semana 2:

- Ubicación: partido, region (derivadas de zona en Bronze)
- Precio: tipo_operacion, precio, moneda, expensas, precio_por_m2
- Características: ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, antiguedad, cochera, orientacion, estado, url, fecha_publicacion
- Metadata: _source_table, _processing_timestamp

In [0]:
-- Ejercicio 2.2
-- Crear Tabla Silver
DROP TABLE IF EXISTS bootcamp_de_valentin.silver.propiedades_silver;
CREATE TABLE bootcamp_de_valentin.silver.propiedades_silver
(
  propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Id unico de la propiedad (auto-generado)',

  partido STRING COMMENT 'Partido/Municipio estandarizado',
  region STRING COMMENT 'Region geografica: capital federal, gba zona norte/oeste/sur',

  tipo_operacion STRING COMMENT 'Alquiler, venta, alquiler_temporario',
  precio DECIMAL(15,2) COMMENT 'Precio de la propiedad',
  moneda STRING COMMENT 'USD o ARS',
  expensas DECIMAL(15,2) COMMENT 'Expensas mensuales',
  precio_por_m2 DECIMAL(15,2) COMMENT 'Precio por metro cuadrado',

  ambientes INT COMMENT 'Cantidad de ambientes',
  m2_totales DECIMAL(15,2) COMMENT 'Superficie total',
  m2_cubiertos DECIMAL(15,2) COMMENT 'Superficie cubierta',
  antiguedad INT COMMENT 'Años de antiguedad (999 si no disponible)',
  cochera BOOLEAN COMMENT 'Tiene cochera o no',
  orientacion STRING COMMENT 'Orientacion del inmueble',
  estado STRING COMMENT 'Estado de la propiedad',
  url STRING COMMENT 'URL de la propiedad',
  fecha_publicacion DATE COMMENT 'Fecha de publicacion - desde fecha (STRING) en Bronze',

  _source_table STRING DEFAULT 'bootcamp_de_valentin.bronze.propiedades_bronze' COMMENT 'Tabla de origen',
  _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP() COMMENT 'Timestamp de procesamiento'
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Propiedades inmobiliarias - Capa Silver (datos limpios y validados)';

In [0]:
-- Ejercicio 2.3
-- Verificacion de estructura creada
DESCRIBE EXTENDED bootcamp_de_valentin.silver.propiedades_silver

# 3. Transformacion Bronze --> Silver

In [0]:
-- Ejercicio 3.1 
-- Ejecutar limpieza de Semana 2
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean AS
SELECT
  CASE WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double ELSE NULL END AS precio,
  moneda,
  CASE WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double ELSE NULL END AS ambientes,
  CASE WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double ELSE NULL END AS m2_totales,
  CASE WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double ELSE NULL END AS m2_cubiertos,
  CASE WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double ELSE NULL END AS antiguedad,
  tipo_de_operacion,
  id,
  ubicacion,
  numero,
  calle,
  expensas,
  orientacion_cardinal,
  orientacion_inmueble,
  piso,
  cochera,
  estado,
  tipo_vendedor,
  url,
  zona,
  fecha,
  hora
FROM bootcamp_de_valentin.bronze.propiedades_bronze;

-- propiedades_clean_2: filtro de outliers por percentiles (Semana 2)
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
  WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE 
      precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT 
    p.*
  FROM propiedades_clean p
  JOIN limites l
    ON p.moneda = l.moneda
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE 
    p.precio BETWEEN l.p01 AND l.p99
);

-- bronze_EDA: limpieza detallada (Semana 2)
CREATE OR REPLACE TEMP VIEW bronze_EDA AS (
  SELECT
    edl.id,
    edl.ubicacion,
    CASE
      WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.precio::float BETWEEN 0 AND 2147483648 THEN edl.precio::float
      ELSE NULL
    END AS precio,
    CASE
      WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
      ELSE NULL
    END AS numero,
    edl.calle,
    CASE
      WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.expensas::float BETWEEN 0 AND 20000000 THEN edl.expensas::float
      ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
      WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
      WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
      WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
      WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
      ELSE edl.moneda
    END AS moneda,
    CASE
      WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.ambientes::float
    END AS ambientes,
    CASE
      WHEN edl.m2_totales = 'NaN' OR edl.m2_totales NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.m2_totales::decimal
    END AS m2_totales,
    CASE
      WHEN edl.m2_cubiertos = 'NaN' OR edl.m2_cubiertos NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.m2_cubiertos::decimal
    END AS m2_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
      WHEN edl.piso IS NULL THEN NULL
      WHEN edl.piso = 'NaN' THEN NULL
      ELSE edl.piso::float
    END AS piso,
    CASE
      WHEN edl.cochera = 'tiene' THEN 1
      ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    COALESCE(CAST(edl.fecha AS DATE), CURRENT_DATE()) AS fecha
  FROM propiedades_clean_2 edl
  WHERE 
    edl.url RLIKE 'https'
    AND (edl.zona LIKE '%gba%' OR edl.zona LIKE '%caba%' OR edl.zona LIKE '%capital%')
    AND edl.zona LIKE '%-%'
    AND LENGTH(edl.zona) < 50
);

SELECT COUNT(*) as registros_bronze_eda FROM bronze_EDA;

In [0]:
-- Ejercicio 3.2
-- Pipeline Bronze_EDA --> Silver